<a href="https://colab.research.google.com/github/chongal/mytravelsolution/blob/new-feature/Final(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Information Retrieval

### 2GIS

In [ ]:
# @title 2GIS packets installation
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo apt-key add -
!sudo sh -c 'echo "deb https://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list'
!sudo apt-get update
!sudo apt-get install google-chrome-stable

!sudo apt-get install -f
!sudo apt-get install mlocate
!pip install parser_2gis

!locate -b google-chrome-stable | fgrep -w bin

OK
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,215 B]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,812 kB]
Hit:14 https://ppa.laun

In [ ]:
# @title 2GIS organization collection
import requests
import json
from typing import List, Dict, Optional

class TwoGISSearcher:
    """
    A class to search for organizations in 2GIS API by category and location.

    API Documentation: https://docs-new.2gis.com/en
    """

    def __init__(self, api_key: str):
        """
        Initialize the 2GIS searcher.

        Args:
            api_key: Your 2GIS API key (get it from Platform Manager)
        """
        self.api_key = api_key
        self.base_url = "https://catalog.api.2gis.com"

    def search_categories(self, category_query: str, region_id: Optional[str] = None) -> List[Dict]:
        """
        Search for categories (rubrics) by name.

        Args:
            category_query: Search query for category (e.g., "cafe", "hotel", "restaurant")
            region_id: Optional region ID to limit search (e.g., "32" for Moscow)

        Returns:
            List of categories with their IDs and names
        """
        url = f"{self.base_url}/2.0/catalog/rubric/search"
        params = {
            'q': category_query,
            'key': self.api_key
        }

        if region_id:
            params['region_id'] = region_id

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()

            if data['meta']['code'] == 200:
                return data['result']['items']
            else:
                print(f"Error: {data['meta'].get('error', {}).get('message', 'Unknown error')}")
                return []
        except Exception as e:
            print(f"Error searching categories: {e}")
            return []

    def search_organizations(self,
                           rubric_id: Optional[str] = None,
                           query: Optional[str] = None,
                           point: Optional[str] = None,
                           radius: Optional[int] = None,
                           city_id: Optional[str] = None,
                           region_id: Optional[str] = None,
                           page: int = 1,
                           page_size: int = 50) -> Dict:
        """
        Search for organizations by category and location.

        Args:
            rubric_id: Category ID(s), can be comma-separated for multiple (e.g., "162,350")
            query: Text search query (optional)
            point: Center point coordinates as "lon,lat" (e.g., "37.622504,55.753215")
            radius: Search radius in meters (e.g., 5000)
            city_id: City ID to limit search
            region_id: Region ID to limit search
            page: Page number (default: 1)
            page_size: Results per page (default: 50, max varies by endpoint)

        Returns:
            Dictionary with search results and metadata
        """
        url = f"{self.base_url}/3.0/items"

        # Build parameters
        params = {
            'key': self.api_key,
            'page': page,
            'page_size': page_size,
            'type': 'branch',  # Get branches/organizations
            'fields': ','.join([
                'items.point',              # Location coordinates
                'items.address',            # Full address
                'items.contact_groups',     # Contact information
                'items.reviews',            # Reviews and ratings
                'items.schedule',           # Working hours
                'items.rubrics',            # Categories
                'items.external_content',   # Website and social media
                'items.org',                # Organization details
                'items.purpose',            # Purpose/description
                'items.attribute_groups',   # Additional attributes
                'items.context',            # Context information
                'items.stat',               # Statistics
                'items.flags',              # Flags (e.g., is_paid)
            ])
        }

        # Add search criteria
        if rubric_id:
            params['rubric_id'] = rubric_id
        if query:
            params['q'] = query
        if point:
            params['point'] = point
            params['sort_point'] = point
            params['sort'] = 'distance'
        if radius:
            params['radius'] = radius
        if city_id:
            params['city_id'] = city_id
        if region_id:
            params['region_id'] = region_id

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()

            if data['meta']['code'] == 200:
                return data['result']
            else:
                print(f"Error: {data['meta'].get('error', {}).get('message', 'Unknown error')}")
                return {'items': [], 'total': 0}
        except Exception as e:
            print(f"Error searching organizations: {e}")
            return {'items': [], 'total': 0}

    def get_region_by_coordinates(self, lon: float, lat: float) -> Optional[Dict]:
        """
        Get region information by coordinates.

        Args:
            lon: Longitude
            lat: Latitude

        Returns:
            Region information
        """
        url = f"{self.base_url}/2.0/region/search"
        params = {
            'q': f"{lon},{lat}",
            'key': self.api_key
        }

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()

            if data['meta']['code'] == 200 and data['result']['items']:
                return data['result']['items'][0]
            return None
        except Exception as e:
            print(f"Error getting region: {e}")
            return None

    def __format_organization(self, org: Dict) -> Dict:
        """
        Format organization data into a readable structure.

        Args:
            org: Raw organization data from API

        Returns:
            Formatted organization data
        """
        formatted = {
            'id': org.get('id'),
            'name': org.get('name'),
            'type': org.get('type'),
            'address': None,
            'location': None,
            'contacts': [],
            'rating': None,
            'reviews_count': None,
            'schedule': None,
            'website': None,
            'social_media': [],
            'description': None,
            'categories': [],
            'attributes': []
        }

        # Address
        if 'address_name' in org:
            formatted['address'] = org['address_name']
            if org.get('address_comment'):
                formatted['address'] += f" ({org['address_comment']})"

        # Location coordinates
        if 'point' in org:
            formatted['location'] = {
                'latitude': org['point']['lat'],
                'longitude': org['point']['lon']
            }

        # Contacts
        if 'contact_groups' in org:
            for group in org['contact_groups']:
                for contact in group.get('contacts', []):
                    contact_info = {
                        'type': contact.get('type'),
                        'value': contact.get('value'),
                        'text': contact.get('text'),
                        'comment': contact.get('comment')
                    }
                    formatted['contacts'].append(contact_info)

        # Reviews and rating
        if 'reviews' in org:
            reviews = org['reviews']
            formatted['rating'] = reviews.get('rating')
            formatted['reviews_count'] = reviews.get('general_review_count', 0)

        # Schedule
        if 'schedule' in org:
            formatted['schedule'] = org['schedule']

        # External content (website, social media)
        if 'external_content' in org:
            for content in org['external_content']:
                if content.get('type') == 'common':
                    formatted['website'] = content.get('url')
                elif content.get('type') in ['social', 'social_network']:
                    formatted['social_media'].append({
                        'type': content.get('subtype'),
                        'url': content.get('url')
                    })

        # Description/purpose
        if 'purpose' in org:
            formatted['description'] = org['purpose'].get('name')

        # Categories (rubrics)
        if 'rubrics' in org:
            formatted['categories'] = [r.get('name') for r in org['rubrics']]

        # Additional attributes
        if 'attribute_groups' in org:
            for group in org['attribute_groups']:
                for attr in group.get('attributes', []):
                    formatted['attributes'].append({
                        'name': attr.get('name'),
                        'value': attr.get('tag_name')
                    })

        return org

    def format_organization(self, org: Dict) -> Dict:
        """
        Format organization data into a readable structure.

        Args:
            org: Raw organization data from API

        Returns:
            Formatted organization data
        """
        formatted = {
            'id': org.get('id'),
            'name': org.get('name'),
            'type': org.get('type'),
            'address': None,
            'location': None,
            'contacts': [],
            'rating': None,
            'reviews_count': None,
            'schedule': None,
            'website': None,
            'social_media': [],
            'description': None,
            'categories': [],
            'attributes': []
        }

        # Address
        if 'address_name' in org:
            formatted['address'] = org['address_name']
            if org.get('address_comment'):
                formatted['address'] += f" ({org['address_comment']})"

        # Location coordinates
        if 'point' in org:
            formatted['location'] = {
                'latitude': org['point']['lat'],
                'longitude': org['point']['lon']
            }

        # Contacts
        if 'contact_groups' in org:
            for group in org['contact_groups']:
                for contact in group.get('contacts', []):
                    contact_info = {
                        'type': contact.get('type'),
                        'value': contact.get('value'),
                        'text': contact.get('text'),
                        'comment': contact.get('comment')
                    }
                    formatted['contacts'].append(contact_info)

        # Reviews and rating
        if 'reviews' in org:
            reviews = org['reviews']
            formatted['rating'] = reviews.get('rating')
            formatted['reviews_count'] = reviews.get('general_review_count', 0)

        # Schedule
        if 'schedule' in org:
            formatted['schedule'] = org['schedule']

        # External content (website, social media)
        if 'external_content' in org:
            for content in org['external_content']:
                if content.get('type') == 'common':
                    formatted['website'] = content.get('url')
                elif content.get('type') in ['social', 'social_network']:
                    formatted['social_media'].append({
                        'type': content.get('subtype'),
                        'url': content.get('url')
                    })

        # Description/purpose
        if 'purpose' in org:
            formatted['description'] = org['purpose'].get('name')

        # Categories (rubrics)
        if 'rubrics' in org:
            formatted['categories'] = [r.get('name') for r in org['rubrics']]

        # Additional attributes
        if 'attribute_groups' in org:
            for group in org['attribute_groups']:
                for attr in group.get('attributes', []):
                    formatted['attributes'].append({
                        'name': attr.get('name'),
                        'value': attr.get('tag_name')
                    })

        return formatted

    def print_organization(self, org: Dict):
        """
        Pretty print organization information.

        Args:
            org: Formatted organization data
        """
        print("\n" + "="*80)
        print(f"Name: {org['name']}")
        print(f"ID: {org['id']}")

        if org['address']:
            print(f"Address: {org['address']}")

        if org['location']:
            print(f"Location: {org['location']['latitude']}, {org['location']['longitude']}")

        if org['rating'] is not None:
            print(f"Rating: {org['rating']} ({org['reviews_count']} reviews)")

        if org['contacts']:
            print("\nContacts:")
            for contact in org['contacts']:
                text = contact['text'] or contact['value']
                comment = f" - {contact['comment']}" if contact['comment'] else ""
                print(f"  {contact['type']}: {text}{comment}")

        if org['website']:
            print(f"\nWebsite: {org['website']}")

        if org['social_media']:
            print("\nSocial Media:")
            for social in org['social_media']:
                print(f"  {social['type']}: {social['url']}")

        if org['schedule']:
            print(f"\nSchedule: {org['schedule'].get('name', 'Available')}")

        if org['categories']:
            print(f"\nCategories: {', '.join(org['categories'])}")

        if org['description']:
            print(f"Description: {org['description']}")

        if org['attributes']:
            print("\nAttributes:")
            for attr in org['attributes']:
                print(f"  - {attr['name']}: {attr['value']}")

        print("="*80)


def main():
    """
    Example usage of the 2GIS searcher.
    """
    # Initialize with your API key
    API_KEY = "401b0774-dbe8-4f70-91c9-697adac3e650"  # Replace with your actual API key
    searcher = TwoGISSearcher(API_KEY)

    # Example 1: Search for hotels in Moscow
    print("Searching for categories related to 'hotel'...")
    categories = searcher.search_categories("hotel", region_id="32")  # 32 = Moscow

    if categories:
        print(f"\nFound {len(categories)} categories:")
        for cat in categories[:5]:  # Show first 5
            print(f"  - {cat['name']} (ID: {cat['id']})")

        # Use the first category ID
        rubric_id = categories[0]['id']
        print(f"\nSearching for organizations in category '{categories[0]['name']}'...")

        # Search for hotels near Moscow center
        results = searcher.search_organizations(
            rubric_id=rubric_id,
            point="37.622504,55.753215",  # Moscow center (lon, lat)
            radius=5000,  # 5km radius
            page_size=10
        )

        print(f"\nFound {results['total']} organizations")
        print(f"Showing {len(results['items'])} results:\n")

        for org_data in results['items']:
            formatted_org = searcher.format_organization(org_data)
            #searcher.print_organization(formatted_org)

    # Example 2: Search for restaurants with a text query
    print("\n\n" + "="*80)
    print("Example 2: Searching for restaurants near a location")
    print("="*80)

    results = searcher.search_organizations(
        rubric_id="547", # Базы отдыха
        radius=15000, # 15 km radius
        page_size=2,
        region_id="67" # almaty
    )

    print(f"\nFound {results['total']} restaurants")
    for org_data in results['items']:
        formatted_org = searcher.format_organization(org_data)
        #searcher.print_organization(formatted_org)

    # Example 3: Export to JSON
    print("\n\nExporting results to JSON file...")
    output_data = {
        'total': results['total'],
        'organizations': [searcher.format_organization(org) for org in results['items']]
    }

    with open('2gis_results.json', 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    print("Results exported to '2gis_results.json'")
    return output_data

#if __name__ == "__main__":
#    main()
organizations = main()

Searching for categories related to 'hotel'...
Error: Method is not allowed for the key. Please, send an email at api@2gis.ru.


Example 2: Searching for restaurants near a location

Found 706 restaurants


Exporting results to JSON file...
Results exported to '2gis_results.json'


In [ ]:
# @title 2GIS data collection (FIXED)
import sys, os
from parser_2gis import main as parser_main
import time

def parse_firm_to_csv(city: str, rubric: str, firm_id: str, index: int, num_records=10):
    """
    Парсит данные одной организации из 2GIS

    Args:
        city: Город (например, 'almaty')
        rubric: Категория (например, 'Базы отдыха')
        firm_id: ID организации из 2GIS
        index: Индекс для имени файла
        num_records: Максимум записей для парсинга
    """
    setattr(sys, 'argv', [
        os.path.abspath(os.getcwd()),
        '-i', f'https://2gis.kz/{city}/search/{rubric}/firm/{firm_id}/',
        '-o', f"/content/results_{index}.csv",
        '-f', "csv",
        '--parser.max-records', f'{num_records}',
        '--chrome.headless', 'yes',
    ])

    try:
        parser_main()
        print(f"  ✅ Создан файл: results_{index}.csv")
    except Exception as e:
        print(f"  ❌ Ошибка для firm_id={firm_id}: {e}")


# Проверяем наличие данных organizations
if 'organizations' not in globals():
    print("❌ Переменная 'organizations' не найдена!")
    print("💡 Сначала выполните ячейку '2GIS organization collection'")
else:
    orgs = organizations.get('organizations', [])

    if not orgs:
        print("⚠️ Список организаций пуст!")
    else:
        print(f"🚀 Начинаем парсинг {len(orgs)} организаций...")
        print(f"📍 Город: almaty")
        print(f"📂 Категория: Базы отдыха\n")

        successful = 0
        failed = 0

        for i, org in enumerate(orgs):
            # ИСПРАВЛЕНИЕ: используем org['id'] вместо org.id
            firm_id = org.get('id')
            firm_name = org.get('name', 'Неизвестно')

            if not firm_id:
                print(f"  ⚠️ [{i+1}/{len(orgs)}] Пропуск: нет ID для '{firm_name}'")
                failed += 1
                continue

            print(f"  🔄 [{i+1}/{len(orgs)}] Парсинг: {firm_name} (ID: {firm_id})")

            try:
                parse_firm_to_csv('almaty', 'Базы отдыха', firm_id, i, num_records=10)
                successful += 1

                # Задержка между запросами, чтобы не забанили
                if i < len(orgs) - 1:
                    time.sleep(2)

            except Exception as e:
                print(f"  ❌ Ошибка: {e}")
                failed += 1

        print(f"\n{'='*60}")
        print(f"✅ Парсинг завершён!")
        print(f"{'='*60}")
        print(f"📊 Статистика:")
        print(f"   • Всего организаций: {len(orgs)}")
        print(f"   • Успешно обработано: {successful}")
        print(f"   • Ошибок: {failed}")
        print(f"\n💾 Созданы файлы: results_0.csv, results_1.csv, ...")

18/10/2025 23:02:22.709 | INFO     | Парсинг запущен.
INFO:parser-2gis:Парсинг запущен.
18/10/2025 23:02:22.711 | INFO     | Парсинг ссылки https://2gis.kz/almaty/search/Базы отдыха/firm/70000001041870952/
INFO:parser-2gis:Парсинг ссылки https://2gis.kz/almaty/search/Базы отдыха/firm/70000001041870952/


🚀 Начинаем парсинг 2 организаций...
📍 Город: almaty
📂 Категория: Базы отдыха

  🔄 [1/2] Парсинг: Восемь озёр, центр отдыха (ID: 70000001041870952)


18/10/2025 23:02:38.561 | INFO     | Парсинг [1] > улица Мустафина, 1а/1
INFO:parser-2gis:Парсинг [1] > улица Мустафина, 1а/1
18/10/2025 23:02:38.993 | INFO     | Парсинг [2] > Tau-Asu
INFO:parser-2gis:Парсинг [2] > Tau-Asu
18/10/2025 23:02:39.522 | INFO     | Парсинг [3] > База отдыха Tau Asu
INFO:parser-2gis:Парсинг [3] > База отдыха Tau Asu
18/10/2025 23:02:39.985 | INFO     | Парсинг [4] > Eco-hotel AQBULAQ
INFO:parser-2gis:Парсинг [4] > Eco-hotel AQBULAQ
18/10/2025 23:02:40.347 | INFO     | Парсинг [5] > База отдыха Eco-Hotel AQBULAQ
INFO:parser-2gis:Парсинг [5] > База отдыха Eco-Hotel AQBULAQ
18/10/2025 23:02:41.224 | INFO     | Парсинг [6] > Гора Глэмпинг
INFO:parser-2gis:Парсинг [6] > Гора Глэмпинг
18/10/2025 23:02:41.604 | INFO     | Парсинг [7] > База отдыха Гора Глэмпинг
INFO:parser-2gis:Парсинг [7] > База отдыха Гора Глэмпинг
18/10/2025 23:02:41.921 | INFO     | Парсинг [8] > E`den
INFO:parser-2gis:Парсинг [8] > E`den
18/10/2025 23:02:42.231 | INFO     | Парсинг [9] > База 

  ✅ Создан файл: results_0.csv


18/10/2025 23:02:45.058 | INFO     | Парсинг запущен.
INFO:parser-2gis:Парсинг запущен.
18/10/2025 23:02:45.061 | INFO     | Парсинг ссылки https://2gis.kz/almaty/search/Базы отдыха/firm/70000001068684985/
INFO:parser-2gis:Парсинг ссылки https://2gis.kz/almaty/search/Базы отдыха/firm/70000001068684985/


  🔄 [2/2] Парсинг: Тау-Дастархан, гостинично-ресторанный комплекс для отдыха в горах (ID: 70000001068684985)


18/10/2025 23:02:59.622 | INFO     | Парсинг [1] > улица Алма-Арасан Ущелье, 1/3
INFO:parser-2gis:Парсинг [1] > улица Алма-Арасан Ущелье, 1/3
18/10/2025 23:03:00.226 | INFO     | Парсинг [2] > Tau-Asu
INFO:parser-2gis:Парсинг [2] > Tau-Asu
18/10/2025 23:03:00.659 | INFO     | Парсинг [3] > База отдыха Tau Asu
INFO:parser-2gis:Парсинг [3] > База отдыха Tau Asu
18/10/2025 23:03:01.264 | INFO     | Парсинг [4] > Eco-hotel AQBULAQ
INFO:parser-2gis:Парсинг [4] > Eco-hotel AQBULAQ
18/10/2025 23:03:01.690 | INFO     | Парсинг [5] > Гора Глэмпинг
INFO:parser-2gis:Парсинг [5] > Гора Глэмпинг
18/10/2025 23:03:02.444 | INFO     | Парсинг [6] > База отдыха Eco-Hotel AQBULAQ
INFO:parser-2gis:Парсинг [6] > База отдыха Eco-Hotel AQBULAQ
18/10/2025 23:03:02.837 | INFO     | Парсинг [7] > База отдыха Гора Глэмпинг
INFO:parser-2gis:Парсинг [7] > База отдыха Гора Глэмпинг
18/10/2025 23:03:03.173 | INFO     | Парсинг [8] > E`den
INFO:parser-2gis:Парсинг [8] > E`den
18/10/2025 23:03:03.503 | INFO     | Пар

  ✅ Создан файл: results_1.csv

✅ Парсинг завершён!
📊 Статистика:
   • Всего организаций: 2
   • Успешно обработано: 2
   • Ошибок: 0

💾 Созданы файлы: results_0.csv, results_1.csv, ...


In [ ]:
# @title 2GIS Packet Merging (Максимально Устойчивая Версия)
import pandas as pd
import glob
import os
import json
import numpy as np # Для работы с NaN

# --- Вспомогательные функции для очистки данных ---

def format_contacts(contacts):
    """Преобразует список контактов в JSON-строку для сохранения в CSV."""
    if isinstance(contacts, list) and contacts:
        return json.dumps(contacts, ensure_ascii=False)
    return ''

def format_social_media(social_media):
    """Преобразует список соцсетей в JSON-строку для сохранения в CSV."""
    if isinstance(social_media, list) and social_media:
        return json.dumps(social_media, ensure_ascii=False)
    return ''

# --- Основная функция объединения ---

def merge_2gis_results():
    """
    Объединяет результаты парсинга 2GIS из нескольких CSV файлов (Data Collection)
    и обогащает их данными из Organizations Collection (глобальная organizations).
    """
    print("🔄 Начинаем объединение данных 2GIS...")

    # 1. Собираем все CSV файлы results_*.csv
    csv_files = sorted(glob.glob("/content/results_*.csv"))

    if not csv_files:
        print("⚠️ Не найдено файлов results_*.csv в /content/")
        return pd.DataFrame()

    print(f"📁 Найдено файлов: {len(csv_files)}")

    # 2. Читаем и объединяем все CSV
    all_dataframes = []

    for file in csv_files:
        try:
            # Читаем с гибкими разделителями и кодировкой
            df = pd.read_csv(file, encoding='utf-8', sep=None, engine='python')

            # Приводим названия колонок к нижнему регистру для унификации поиска
            df.columns = df.columns.str.lower()

            print(f"  ✓ Загружен {file}: {len(df)} записей")
            all_dataframes.append(df)
        except Exception as e:
            print(f"  ✗ Ошибка при чтении {file}: {e}")

    if not all_dataframes:
        print("❌ Не удалось загрузить ни одного файла")
        return pd.DataFrame()

    # 3. Объединяем все DataFrame
    merged_df = pd.concat(all_dataframes, ignore_index=True)

    # --- ОПРЕДЕЛЕНИЕ КЛЮЧЕВЫХ КОЛОНОК ---

    # Определяем наиболее вероятный столбец с именем для очистки/дубликатов
    name_cols = [col for col in ['name', 'title', 'organization_name', 'org_name'] if col in merged_df.columns]

    # Используем первый найденный столбец с именем
    name_column = name_cols[0] if name_cols else None

    if name_column:
        # Очистка от строк, где имя организации отсутствует
        merged_df = merged_df.dropna(subset=[name_column]).reset_index(drop=True)
        print(f"  ✓ Используем столбец '{name_column}' для очистки и удаления дубликатов.")
    else:
        print("⚠️ Внимание: Столбец для имени не найден. Пропускаем удаление строк без имени.")

    print(f"\n📊 Всего записей после первичного объединения: {len(merged_df)}")

    # 4. Добавляем URL профилей 2GIS и другие поля из organizations
    organizations = globals().get('organizations')

    if organizations and isinstance(organizations, dict) and 'organizations' in organizations:
        print("\n🔗 Начинаем обогащение данных из organizations (Organization Collection)...")

        # Создаём словарь для быстрого поиска по ID
        org_dict = {}
        for org in organizations.get('organizations', []):
            org_id = org.get('id')
            if org_id:
                # Собираем все нужные данные в словарь
                org_dict[str(org_id)] = {
                    'url': f"https://2gis.kz/almaty/firm/{org_id}",
                    'rating': org.get('rating'),
                    'reviews_count': org.get('reviews_count'),
                    'website': org.get('website'),
                    'location': org.get('location', {}),
                    'contacts': org.get('contacts', []),
                    'social_media': org.get('social_media', [])
                }

        # Функция для безопасного получения ID (расширенный список поиска)
        def get_org_id(row):
            # Пробуем максимально широкий набор вариантов названий колонок
            for col_name in ['id', 'firm_id', 'organization_id', 'org_id', '2gis_id']:
                if col_name in row.index and pd.notna(row[col_name]):
                    try:
                        # Безопасное приведение к строке: сначала int, потом str
                        if pd.api.types.is_numeric_dtype(row[col_name]):
                             # Учитываем, что ID могут быть очень длинными (как в 2GIS)
                            return str(int(row[col_name]))
                        else:
                            return str(row[col_name])
                    except ValueError:
                         return str(row[col_name])
            return None

        # Извлекаем Instagram URL
        def extract_instagram(org_id):
            if not org_id: return ''
            social = org_dict.get(org_id, {}).get('social_media', [])
            for s in social:
                url = s.get('url', '').lower()
                if 'instagram' in url and url.startswith('http'):
                    return s.get('url')
            return ''

        # Добавляем временный столбец с найденным ID
        merged_df['temp_org_id'] = merged_df.apply(get_org_id, axis=1)

        # Обогащение данных через map/apply
        merged_df['profile_url'] = merged_df['temp_org_id'].apply(lambda x: org_dict.get(x, {}).get('url', '') if x else '')
        merged_df['rating_2gis'] = merged_df['temp_org_id'].apply(lambda x: org_dict.get(x, {}).get('rating', np.nan) if x else np.nan)
        merged_df['reviews_count_2gis'] = merged_df['temp_org_id'].apply(lambda x: org_dict.get(x, {}).get('reviews_count', 0) if x else 0)
        merged_df['website'] = merged_df['temp_org_id'].apply(lambda x: org_dict.get(x, {}).get('website', '') if x else '')

        merged_df['instagram_url'] = merged_df['temp_org_id'].apply(extract_instagram)

        merged_df['latitude'] = merged_df['temp_org_id'].apply(lambda x: org_dict.get(x, {}).get('location', {}).get('latitude', np.nan) if x else np.nan)
        merged_df['longitude'] = merged_df['temp_org_id'].apply(lambda x: org_dict.get(x, {}).get('location', {}).get('longitude', np.nan) if x else np.nan)
        merged_df['contacts_json'] = merged_df['temp_org_id'].apply(lambda x: format_contacts(org_dict.get(x, {}).get('contacts', [])) if x else '')
        merged_df['social_media_json'] = merged_df['temp_org_id'].apply(lambda x: format_social_media(org_dict.get(x, {}).get('social_media', [])) if x else '')

        # Удаляем временный ID
        merged_df = merged_df.drop(columns=['temp_org_id'], errors='ignore')

        enrichment_count = merged_df['profile_url'].astype(bool).sum()
        print(f"  ✓ Успешно обогащено организаций (найден ID): {enrichment_count} / {len(merged_df)}")
        print(f"  ✓ Добавлены ключевые поля: profile_url, instagram_url, website, rating, coordinates и др.")

    else:
        print("\n⚠️ Внимание: Глобальная переменная 'organizations' (Organization Collection) не найдена или имеет неверный формат. Обогащение пропущено.")

    # 5. Удаляем дубликаты
    original_count = len(merged_df)

    if name_column:
        # Используем найденный столбец для удаления дубликатов
        merged_df = merged_df.drop_duplicates(subset=[name_column], keep='first')
        duplicates_removed = original_count - len(merged_df)
        if duplicates_removed > 0:
            print(f"\n🗑️ Удалено дубликатов по '{name_column}': {duplicates_removed}")
    else:
        print("\n⚠️ Пропуск удаления дубликатов: столбец имени не найден.")

    # 6. Сохраняем результат
    output_file = "/content/merged_2gis_data.csv"
    merged_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\n✅ Данные объединены и сохранены в {output_file}")

    # 7. Выводим статистику
    print(f"\n📈 Статистика финального датасета:")
    print(f"   • Всего уникальных организаций: {len(merged_df)}")
    if 'website' in merged_df.columns:
        print(f"   • Со ссылками на сайт: {merged_df['website'].astype(bool).sum()}")
    if 'instagram_url' in merged_df.columns:
        print(f"   • С Instagram URL: {(merged_df['instagram_url'] != '').sum()}")
    if 'rating_2gis' in merged_df.columns:
        print(f"   • С рейтингом 2GIS: {merged_df['rating_2gis'].notna().sum()}")

    # 8. Показываем пример данных
    print("\n📋 Пример данных (первые 3 строки):")
    # Добавляем найденный столбец имени в список отображения, если он есть
    display_cols = [name_column] if name_column else []
    display_cols.extend(['profile_url', 'rating_2gis', 'reviews_count_2gis', 'website', 'instagram_url'])

    available_cols = [col for col in display_cols if col in merged_df.columns]
    print(merged_df[available_cols].head(3).to_string())

    return merged_df

# Запускаем объединение
merged_2gis_data = merge_2gis_results()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



🔄 Загружаем сохраненную сессию...

❌ Критическая ошибка авторизации: 'Client' object has no attribute 'start_session'

💡 Скрипт остановлен. Проверьте логин/пароль/2FA.
Traceback (most recent call last):
  File "/tmp/ipython-input-3263539525.py", line 31, in <cell line: 0>
    cl.start_session()
    ^^^^^^^^^^^^^^^^
AttributeError: 'Client' object has no attribute 'start_session'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3263539525.py", line 45, in <cell line: 0>
    sys.exit() # Останавливаем выполнение ячейки при фатальной ошибке
    ^^^^^^^^^^
SystemExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/ultratb.py", line 110

TypeError: object of type 'NoneType' has no len()

### Instagram

In [ ]:
# @title Packets installation
##!pip install instagrapi
##!pip uninstall instagrapi -y
!pip install instagrapi==1.17.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.3/143.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 8.7 MB/s eta 0:00:00
  Created wheel for instagrapi: filename=instagrapi-1.17.1-py3-none-any.whl size=101020 sha256=dc6be671c53ecdbfda163aaa45280d6436ea876d5fcffb8c4f1dedd5b17dc42a
  Stored in directory: /root/.cache/pip/wheels/ed/41/af/a931c43d0d644df318b9f1cf261468a0df96e80c364591cd9b
Successfully built instagrapi
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.24
    Uninstalling pydantic-1.10.24:
      Successfully uninstalled pydantic-1.10.24
  Attempting uninstall: pycryptodomex
    Found existing installation: pycryptodomex 3.23.0
    Uninstalling pycryptodomex-3.23.0:
      Successfully uninstalled pycryptodomex-3.

In [ ]:
# ============================================================
# ✅ INSTAGRAM DATA INTEGRATION (STABLE, FIXED FOR "instagram" COLUMN)
# ============================================================

import pandas as pd
import re
from datetime import datetime, timedelta, timezone
import time
import os
import sys

# === ПРОВЕРКА ГЛОБАЛЬНОГО КЛИЕНТА ===
if 'cl' not in globals() or globals()['cl'] is None or not hasattr(globals()['cl'], 'user_id'):
    print("❌ Глобальная переменная 'cl' (клиент Instagram) не найдена или не авторизована!")
    print("💡 Сначала выполните ячейку '1. Instagram Client Initialization & Test'.")
    sys.exit()
else:
    instagram_client = globals()['cl']
    print("✅ Используем существующую глобальную сессию Instagram.")


# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

def extract_username_from_url(instagram_url: str):
    """
    Извлекает username из разных форм Instagram URL или @username.
    Примеры:
      - https://instagram.com/tau_asu_almaty → tau_asu_almaty
      - instagram.com/tau_asu_almaty/ → tau_asu_almaty
      - @tau_asu_almaty → tau_asu_almaty
    """
    if not isinstance(instagram_url, str) or not instagram_url.strip():
        return None

    url = instagram_url.strip()

    # убираем возможные пробелы и мусор
    url = url.replace("?", "").replace("&", "")

    # ищем паттерны URL
    match = re.search(r"(?:https?:\/\/)?(?:www\.)?instagram\.com\/([A-Za-z0-9_.]+)", url)
    if match:
        return match.group(1)

    # если просто @username
    if url.startswith("@"):
        return url[1:]

    # если просто имя без https
    if re.match(r"^[A-Za-z0-9._]+$", url):
        return url

    return None


def fetch_instagram_stats(username, cl, days_back=30):
    """Получает статистику Instagram для одного аккаунта."""
    cl.public_request_timeout = 5

    try:
        user = cl.user_info_by_username(username)
        cutoff_date = datetime.now(timezone.utc) - timedelta(days=days_back)
        user_id = user.pk
        all_posts = []

        medias = cl.user_medias_v1(user_id, amount=10)  # ограничим запрос
        for post in medias:
            post_date = post.taken_at.replace(tzinfo=timezone.utc)
            if post_date >= cutoff_date:
                all_posts.append({'likes': post.like_count, 'comments': post.comment_count})
            else:
                break

        total_likes = sum(p['likes'] for p in all_posts)
        total_comments = sum(p['comments'] for p in all_posts)
        posts_count = len(all_posts)
        avg_likes = total_likes / posts_count if posts_count > 0 else 0
        avg_comments = total_comments / posts_count if posts_count > 0 else 0
        engagement_rate = (
            round((total_likes + total_comments) / user.follower_count * 100, 2)
            if user.follower_count > 0 else 0
        )

        return {
            'instagram_username': username,
            'instagram_followers': user.follower_count,
            'instagram_posts_total': user.media_count,
            'instagram_posts_last_month': posts_count,
            'instagram_likes_last_month': total_likes,
            'instagram_comments_last_month': total_comments,
            'instagram_avg_likes': round(avg_likes, 2),
            'instagram_avg_comments': round(avg_comments, 2),
            'instagram_engagement_rate': engagement_rate,
            'instagram_fetch_error': None
        }

    except Exception as e:
        error_msg = str(e).split('\n')[0]
        sys.stderr.write(f"❌ Ошибка @{username}: {error_msg}\n")
        return {
            'instagram_username': username,
            'instagram_followers': 0,
            'instagram_posts_total': 0,
            'instagram_posts_last_month': 0,
            'instagram_likes_last_month': 0,
            'instagram_comments_last_month': 0,
            'instagram_avg_likes': 0,
            'instagram_avg_comments': 0,
            'instagram_engagement_rate': 0,
            'instagram_fetch_error': error_msg
        }


def integrate_instagram_data(instagram_client,
                             merged_2gis_csv="/content/final_merged_data.csv",
                             output_csv="/content/final_instagram_enriched.csv",
                             days_back=30,
                             delay_between_requests=10):

    print("🚀 Начинаем интеграцию данных Instagram...")

    if not os.path.exists(merged_2gis_csv):
        print(f"❌ Файл {merged_2gis_csv} не найден!")
        return None

    df = pd.read_csv(merged_2gis_csv)

    # 🔍 Автоматическое определение колонки с Instagram URL
    insta_col = None
    for col in df.columns:
        if "insta" in col.lower():
            insta_col = col
            break

    if insta_col is None:
        print("❌ Не найдена колонка с Instagram URL!")
        print(f"🧾 Найдены колонки: {list(df.columns)}")
        return None

    df_to_process = df[df[insta_col].astype(bool)].copy()
    instagram_count = len(df_to_process)

    instagram_columns = [
        'instagram_username', 'instagram_followers', 'instagram_posts_total',
        'instagram_posts_last_month', 'instagram_likes_last_month', 'instagram_comments_last_month',
        'instagram_avg_likes', 'instagram_avg_comments', 'instagram_engagement_rate',
        'instagram_fetch_error'
    ]
    for col in instagram_columns:
        if col not in df.columns:
            df[col] = None

    print(f"✅ Загружено {len(df)} организаций из 2GIS")
    print(f"📸 Найдено организаций с Instagram: {instagram_count}")
    print(f"⏱️ Задержка между запросами: {delay_between_requests} сек\n")

    processed = 0
    successful = 0

    for idx, row in df_to_process.iterrows():
        username = extract_username_from_url(row[insta_col])

        if not username:
            df.loc[idx, 'instagram_fetch_error'] = "URL Parsing Error"
            continue

        processed += 1
        print(f"🔄 [{processed}/{instagram_count}] Обработка @{username}...")

        stats = fetch_instagram_stats(username, instagram_client, days_back)
        for key, value in stats.items():
            df.loc[idx, key] = value

        if stats and stats['instagram_followers'] > 0:
            successful += 1
            print(
                f"   ✅ Подписчиков: {stats['instagram_followers']:,} | "
                f"Постов/мес: {stats['instagram_posts_last_month']} | "
                f"Engagement: {stats['instagram_engagement_rate']}%"
            )

        if processed < instagram_count:
            time.sleep(delay_between_requests)

    # 💾 Сохранение
    df.to_csv(output_csv, index=False, encoding='utf-8')

    # 📊 Итог
    print(f"\n{'=' * 60}")
    print(f"✅ Интеграция завершена!")
    print(f"📊 Статистика:")
    print(f"   • Всего организаций: {len(df)}")
    print(f"   • С Instagram URL: {instagram_count}")
    print(f"   • Обработано успешно: {successful}")
    print(f"   • С ошибкой: {instagram_count - successful}")
    print(f"💾 Данные сохранены в: {output_csv}")

    # 📋 Пример успешных строк
    available_cols = [
        'name', 'instagram_username', 'instagram_followers',
        'instagram_posts_last_month', 'instagram_avg_likes',
        'instagram_engagement_rate', 'instagram_fetch_error'
    ]
    available_cols = [col for col in available_cols if col in df.columns]
    sample_df = df[
        (df['instagram_followers'].notna()) &
        (df['instagram_followers'] > 0)
    ][available_cols].head(3)

    if not sample_df.empty:
        print("\n📋 Пример успешных результатов:")
        print(sample_df.to_string(index=False))

    return df


# 🎯 ЗАПУСК
final_data = integrate_instagram_data(
    instagram_client,
    merged_2gis_csv="/content/final_merged_data.csv",   # <-- у тебя этот файл
    output_csv="/content/final_instagram_enriched.csv",
    days_back=30,
    delay_between_requests=10
)


✅ Используем существующую глобальную сессию Instagram.
🚀 Начинаем интеграцию данных Instagram...
✅ Загружено 20 организаций из 2GIS
📸 Найдено организаций с Instagram: 20
⏱️ Задержка между запросами: 10 сек

🔄 [1/20] Обработка @tau_asu_almaty...


ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/tau_asu_almaty/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/tau_asu_almaty/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/tau_asu_almaty/?__a=1&__d=dis) >>> 
/tmp/ipython-input-2543767727.py:172: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'tau_asu_almaty' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, key] = value


   ✅ Подписчиков: 48,937 | Постов/мес: 0 | Engagement: 0.0%
🔄 [2/20] Обработка @eco_hotel_aqbulaq...


ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/eco_hotel_aqbulaq/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/eco_hotel_aqbulaq/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/eco_hotel_aqbulaq/?__a=1&__d=dis) >>> 


   ✅ Подписчиков: 13,780 | Постов/мес: 0 | Engagement: 0.0%
🔄 [3/20] Обработка @gora.glamping...


...

   ✅ Подписчиков: 21,860 | Постов/мес: 1 | Engagement: 0.14%
🔄 [4/20] Обработка @eden_mountain_house...


ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/eden_mountain_house/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/eden_mountain_house/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/eden_mountain_house/?__a=1&__d=dis) >>> 


   ✅ Подписчиков: 42,418 | Постов/мес: 4 | Engagement: 0.75%
🔄 [5/20] Обработка @roza.ecovillage...


ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/roza.ecovillage/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/roza.ecovillage/?__a=1&__d=dis) >>> 
ERROR:public_request:Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/roza.ecovillage/?__a=1&__d=dis) >>> 


   ✅ Подписчиков: 1,914 | Постов/мес: 10 | Engagement: 15.67%
🔄 [6/20] Обработка @tau_asu_almaty...
   ✅ Подписчиков: 48,937 | Постов/мес: 0 | Engagement: 0.0%
🔄 [7/20] Обработка @eco_hotel_aqbulaq...
   ✅ Подписчиков: 13,780 | Постов/мес: 0 | Engagement: 0.0%
🔄 [8/20] Обработка @gora.glamping...
   ✅ Подписчиков: 21,860 | Постов/мес: 1 | Engagement: 0.14%
🔄 [9/20] Обработка @eden_mountain_house...
   ✅ Подписчиков: 42,418 | Постов/мес: 4 | Engagement: 0.75%
🔄 [10/20] Обработка @roza.ecovillage...
   ✅ Подписчиков: 1,914 | Постов/мес: 10 | Engagement: 15.67%

✅ Интеграция завершена!
📊 Статистика:
   • Всего организаций: 20
   • С Instagram URL: 20
   • Обработано успешно: 10
   • С ошибкой: 10
💾 Данные сохранены в: /content/final_instagram_enriched.csv

📋 Пример успешных результатов:
instagram_username  instagram_followers  instagram_posts_last_month  instagram_avg_likes  instagram_engagement_rate instagram_fetch_error
    tau_asu_almaty              48937.0                         0.0

## AI analysis

### Ollama installation

In [ ]:
# @title Components installation
!curl https://ollama.ai/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  57402      0 --:--:-- --:--:-- --:--:-- 57245
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://dl.google.com/linux/chrome/deb stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https:/

In [ ]:
# @title Start server
import subprocess
proccess = subprocess.Popen(['ollama', 'serve'])

In [ ]:
# @title Install Models
!ollama pull qwen3:14b

In [ ]:
# @title Model selection
model = "qwen3:14b" # @param ["deepseek-r1:1.5b","deepseek-r1:7b","deepseek-r1:14b","deepseek-r1:32b","deepseek-r1:70b","deepseek-coder:1.3b","deepseek-coder:6.7b","deepseek-coder:33b","gemma3:12b","gemma3:27b","llama3.3:70b","mistral:7b","phi4:14b","qwen2.5:7b","qwen2.5:14b","qwen2.5:32b","qwen2.5-coder:7b","qwen2.5-coder:14b","qwen2.5-coder:32b", "qwen3:14b", "llama3.2:3b"]
!ollama pull {model}

### Model interaction

In [ ]:
!pip uninstall -y pydantic pydantic-core
!pip install -U "pydantic==2.10.5" "pydantic-core==2.23.4"
!pip install -U ollama

Found existing installation: pydantic 2.12.3
Uninstalling pydantic-2.12.3:
  Successfully uninstalled pydantic-2.12.3
Found existing installation: pydantic_core 2.41.4
Uninstalling pydantic_core-2.41.4:
  Successfully uninstalled pydantic_core-2.41.4
  Using cached pydantic-2.10.5-py3-none-any.whl.metadata (30 kB)
  Using cached pydantic_core-2.23.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
INFO: pip is looking at multiple versions of pydantic to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install pydantic-core==2.23.4 and pydantic==2.10.5 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested pydantic-core==2.23.4
    pydantic 2.10.5 depends on pydantic-core==2.27.2

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict



In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!ollama pull qwen3:14b

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
# @title 🔍 ДИАГНОСТИКА + ИСПРАВЛЕННЫЙ AI Description Merger
import pandas as pd
import ollama
import os
import sys
from tqdm import tqdm

# ============================================================
# ШАГ 1: ДИАГНОСТИКА
# ============================================================
print("="*60)
print("🔍 ДИАГНОСТИКА ПРОБЛЕМЫ")
print("="*60)

# 1. Проверка Ollama
print("\n1️⃣ Проверка Ollama...")
try:
    models = ollama.list()
    print("   ✅ Ollama работает!")
    print(f"   📦 Доступные модели: {[m['name'] for m in models.get('models', [])]}")
except Exception as e:
    print(f"   ❌ Ollama НЕ работает: {e}")
    print("\n   🔧 РЕШЕНИЕ:")
    print("   # Запустите в отдельной ячейке:")
    print("   !curl -fsSL https://ollama.com/install.sh | sh")
    print("   !nohup ollama serve > ollama.log 2>&1 &")
    print("   !sleep 5")
    print("   !ollama pull qwen3:14b")

# 2. Проверка файла
print("\n2️⃣ Проверка входного файла...")
input_file = "/content/final_instagram_enriched.csv"
if os.path.exists(input_file):
    print(f"   ✅ Файл найден: {input_file}")
    try:
        df_test = pd.read_csv(input_file)
        print(f"   📊 Строк: {len(df_test)}, Колонок: {len(df_test.columns)}")
        print(f"   📋 Колонки: {list(df_test.columns[:5])}...")
    except Exception as e:
        print(f"   ❌ Ошибка чтения: {e}")
else:
    print(f"   ❌ Файл НЕ найден: {input_file}")
    print("\n   🔧 РЕШЕНИЕ:")
    print("   - Убедитесь, что предыдущий этап (Instagram Enrichment) завершён")
    print("   - Или измените путь к файлу")

# 3. Проверка зависимостей
print("\n3️⃣ Проверка библиотек...")
try:
    import pandas as pd
    print(f"   ✅ pandas: {pd.__version__}")
except:
    print("   ❌ pandas не установлен")

try:
    import ollama
    print(f"   ✅ ollama установлен")
except:
    print("   ❌ ollama не установлен")
    print("   🔧 Установите: !pip install ollama")

try:
    from tqdm import tqdm
    print(f"   ✅ tqdm установлен")
except:
    print("   ❌ tqdm не установлен")
    print("   🔧 Установите: !pip install tqdm")

print("\n" + "="*60)
print("Если все проверки прошли ✅, запускайте код ниже")
print("="*60 + "\n")


# ============================================================
# ИСПРАВЛЕННЫЙ КОД
# ============================================================
class DescriptionMerger:
    """
    Объединяет описания из 2GIS, Website и Instagram в единое AI-описание
    """

    def __init__(self, model_name="qwen3:14b"):
        self.model_name = model_name

        # Проверяем доступность модели
        try:
            models = ollama.list()
            available = [m['name'] for m in models.get('models', [])]
            if model_name not in available:
                print(f"⚠️ Модель {model_name} не найдена!")
                print(f"   Доступны: {available}")
                if available:
                    self.model_name = available[0]
                    print(f"   ✅ Используем: {self.model_name}")
        except Exception as e:
            print(f"⚠️ Не удалось проверить модели: {e}")

        print(f"✅ DescriptionMerger инициализирован с моделью: {self.model_name}\n")

    def safe_get(self, row, key, default=''):
        """Безопасное получение значения"""
        try:
            value = row.get(key, default)
            if pd.isna(value) or value == '' or value == 'N/A':
                return default
            return str(value)
        except:
            return default

    def create_merged_description(self, row):
        """Создаёт AI описание для одной организации"""
        sources = []

        # 1. Данные из 2GIS
        gis_info = []
        name = self.safe_get(row, 'name')
        if name:
            gis_info.append(f"Название: {name}")

        address = self.safe_get(row, 'address')
        if address:
            gis_info.append(f"Адрес: {address}")

        # Рейтинг
        rating = self.safe_get(row, 'rating_2gis')
        if rating:
            try:
                rating_val = float(rating)
                reviews = int(self.safe_get(row, 'reviews_count_2gis', 0))
                gis_info.append(f"Рейтинг: {rating_val}/5 ({reviews} отзывов)")
            except:
                pass

        # Категории
        for col in ['categories', 'рубрики', 'category']:
            categories = self.safe_get(row, col)
            if categories:
                gis_info.append(f"Категории: {categories}")
                break

        # Описание из 2GIS
        for col in ['description', 'описание']:
            desc = self.safe_get(row, col)
            if desc:
                gis_info.append(f"Описание: {desc}")
                break

        if gis_info:
            sources.append("2GIS:\n" + "\n".join(gis_info))

        # 2. Website
        website = self.safe_get(row, 'website_summary')
        if website:
            sources.append(f"Сайт:\n{website}")

        # 3. Instagram
        instagram_info = []
        followers = self.safe_get(row, 'instagram_followers')
        if followers:
            try:
                f = int(followers)
                if f > 0:
                    instagram_info.append(f"Подписчики: {f:,}")
                    engagement = self.safe_get(row, 'instagram_engagement_rate')
                    if engagement:
                        instagram_info.append(f"Вовлечённость: {float(engagement):.2f}%")
            except:
                pass

        if instagram_info:
            sources.append("Instagram:\n" + "\n".join(instagram_info))

        if not sources:
            return "Недостаточно данных"

        # Промпт
        prompt = f"""Создай краткое описание (3-5 предложений) для туристической платформы.

Данные:
{chr(10).join(sources)}

Требования:
- Русский язык
- Продающий стиль
- Укажи ключевые преимущества
- БЕЗ префиксов типа "Описание:" или "На основе..."

Описание:"""

        try:
            response = ollama.generate(
                model=self.model_name,
                prompt=prompt,
                options={'temperature': 0.7, 'num_predict': 400}
            )

            desc = response['response'].strip()

            # Очистка
            for prefix in ["Описание:", "На основе", "Вот описание:"]:
                if desc.startswith(prefix):
                    desc = desc[len(prefix):].strip()

            return desc.strip('"\'')

        except Exception as e:
            return f"Ошибка генерации: {str(e)[:100]}"

    def normalize_columns(self, df):
        """Переименовывает колонки"""
        rename = {
            'наименование': 'name',
            'название': 'name',
            'описание': 'description',
            'рубрики': 'categories',
            'адрес': 'address'
        }

        to_rename = {k: v for k, v in rename.items() if k in df.columns}
        if to_rename:
            df = df.rename(columns=to_rename)
            print(f"✅ Переименованы: {list(to_rename.keys())}")

        return df

    def process_dataframe(self, input_csv, output_csv):
        """Основная обработка"""
        print("\n" + "="*60)
        print("🤖 ЗАПУСК AI DESCRIPTION MERGER")
        print("="*60 + "\n")

        # Загрузка
        if not os.path.exists(input_csv):
            print(f"❌ Файл не найден: {input_csv}")
            return None

        try:
            df = pd.read_csv(input_csv)
            print(f"✅ Загружено: {len(df)} организаций")
            print(f"📊 Колонки: {list(df.columns[:8])}...\n")
        except Exception as e:
            print(f"❌ Ошибка чтения: {e}")
            return None

        # Нормализация
        df = self.normalize_columns(df)

        # Проверка Ollama
        try:
            ollama.list()
            print("✅ Ollama доступен\n")
        except Exception as e:
            print(f"❌ Ollama недоступен: {e}")
            return None

        # Генерация
        if 'ai_description' not in df.columns:
            df['ai_description'] = ''

        print("🚀 Генерация описаний...\n")

        success = 0
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="💬 Обработка"):
            name = self.safe_get(row, 'name', f'Org {idx+1}')
            desc = self.create_merged_description(row)
            df.at[idx, 'ai_description'] = desc

            if "Ошибка" not in desc and "Недостаточно" not in desc:
                success += 1

        # Сохранение
        df.to_csv(output_csv, index=False, encoding='utf-8')

        # Статистика
        print(f"\n{'='*60}")
        print(f"📊 РЕЗУЛЬТАТ")
        print(f"{'='*60}")
        print(f"✅ Успешно: {success}/{len(df)} ({success/len(df)*100:.1f}%)")
        print(f"💾 Сохранено: {output_csv}\n")

        # Примеры
        print("📋 ПРИМЕРЫ:\n")
        good = df[~df['ai_description'].str.contains('Ошибка|Недостаточно', na=False)].head(2)
        for _, row in good.iterrows():
            print(f"🏨 {row.get('name', 'N/A')}")
            print(f"   {row['ai_description'][:150]}...\n")

        return df


# ============================================================
# ЗАПУСК
# ============================================================
if __name__ == "__main__":
    merger = DescriptionMerger(model_name="qwen3:14b")

    result = merger.process_dataframe(
        input_csv="/content/final_instagram_enriched.csv",
        output_csv="/content/enriched_organizations.csv"
    )

    if result is not None:
        print("✨ Успешно завершено!")
    else:
        print("❌ Ошибка. Проверьте диагностику выше.")

🔍 ДИАГНОСТИКА ПРОБЛЕМЫ

1️⃣ Проверка Ollama...
   ✅ Ollama работает!
   ❌ Ollama НЕ работает: 'name'

   🔧 РЕШЕНИЕ:
   # Запустите в отдельной ячейке:
   !curl -fsSL https://ollama.com/install.sh | sh
   !nohup ollama serve > ollama.log 2>&1 &
   !sleep 5
   !ollama pull qwen3:14b

2️⃣ Проверка входного файла...
   ✅ Файл найден: /content/final_instagram_enriched.csv
   📊 Строк: 20, Колонок: 49
   📋 Колонки: ['наименование', 'описание', 'рубрики', 'адрес', 'комментарий к адресу']...

3️⃣ Проверка библиотек...
   ✅ pandas: 2.2.2
   ✅ ollama установлен
   ✅ tqdm установлен

Если все проверки прошли ✅, запускайте код ниже

⚠️ Не удалось проверить модели: 'name'
✅ DescriptionMerger инициализирован с моделью: qwen3:14b


🤖 ЗАПУСК AI DESCRIPTION MERGER

✅ Загружено: 20 организаций
📊 Колонки: ['наименование', 'описание', 'рубрики', 'адрес', 'комментарий к адресу', 'почтовый индекс', 'микрорайон', 'район']...

✅ Переименованы: ['наименование', 'описание', 'рубрики', 'адрес']
✅ Ollama доступен


💬 Обработка:  20%|██        | 4/20 [52:27<3:26:43, 775.21s/it]

## Assessment and Outreach


In [ ]:
# @title Organizations assessment
# @title Organizations Assessment & Scoring
import pandas as pd
import numpy as np
import os

class OrganizationScorer:
    """
    Система оценки организаций по критериям для mytravel.kz

    Критерии оценки:
    1. Online Activity (Активность в сети)
    2. Data Completeness (Полнота данных)
    3. Popularity (Популярность)
    4. Occupancy Potential (Потенциал заполняемости)
    5. Target Audience Fit (Соответствие ЦА)
    6. Commercial Potential (Коммерческий потенциал)
    """

    def __init__(self):
        self.weights = {
            'online_activity': 0.25,      # 25%
            'data_completeness': 0.15,    # 15%
            'popularity': 0.20,           # 20%
            'occupancy_potential': 0.15,  # 15%
            'target_audience': 0.10,      # 10%
            'commercial_potential': 0.15  # 15%
        }

    def score_online_activity(self, row):
        """
        Оценка активности в сети (0-100)

        Учитывает:
        - Частота постов в Instagram
        - Количество подписчиков
        - Вовлечённость аудитории
        - Наличие активного сайта
        """
        score = 0

        # Instagram активность (60 баллов)
        if pd.notna(row.get('instagram_followers')) and row['instagram_followers'] > 0:
            # Подписчики (20 баллов)
            followers = row['instagram_followers']
            if followers >= 10000:
                score += 20
            elif followers >= 5000:
                score += 15
            elif followers >= 1000:
                score += 10
            elif followers >= 500:
                score += 5

            # Посты за месяц (20 баллов)
            posts = row.get('instagram_posts_last_month', 0)
            if posts >= 20:
                score += 20
            elif posts >= 10:
                score += 15
            elif posts >= 5:
                score += 10
            elif posts >= 1:
                score += 5

            # Engagement rate (20 баллов)
            engagement = row.get('instagram_engagement_rate', 0)
            if engagement >= 5:
                score += 20
            elif engagement >= 3:
                score += 15
            elif engagement >= 1:
                score += 10
            elif engagement > 0:
                score += 5

        # Наличие сайта (40 баллов)
        if pd.notna(row.get('website')) and row['website']:
            score += 40

        return min(score, 100)

    def score_data_completeness(self, row):
        """
        Оценка полноты данных (0-100)

        Проверяет наличие:
        - Контактов (телефон, email)
        - Описания
        - Фотографий (косвенно через Instagram)
        - Адреса и координат
        """
        score = 0

        # Основные контакты (40 баллов)
        if pd.notna(row.get('phone')) and row['phone']:
            score += 20
        if pd.notna(row.get('email')) and row['email']:
            score += 20

        # Описание (20 баллов)
        if pd.notna(row.get('ai_description')) and len(str(row['ai_description'])) > 50:
            score += 20
        elif pd.notna(row.get('description')) and row['description']:
            score += 10

        # Локация (20 баллов)
        if pd.notna(row.get('address')) and row['address']:
            score += 10
        if pd.notna(row.get('latitude')) and pd.notna(row.get('longitude')):
            score += 10

        # Соцсети и сайт (20 баллов)
        if pd.notna(row.get('website')) and row['website']:
            score += 10
        if pd.notna(row.get('instagram_url')) and row['instagram_url']:
            score += 10

        return min(score, 100)

    def score_popularity(self, row):
        """
        Оценка популярности (0-100)

        Учитывает:
        - Рейтинг на 2GIS
        - Количество отзывов
        - Лайки и комментарии в Instagram
        """
        score = 0

        # Рейтинг 2GIS (40 баллов)
        rating = row.get('rating_2gis', 0)
        if pd.notna(rating):
            if rating >= 4.5:
                score += 40
            elif rating >= 4.0:
                score += 30
            elif rating >= 3.5:
                score += 20
            elif rating >= 3.0:
                score += 10

        # Количество отзывов (30 баллов)
        reviews = row.get('reviews_count_2gis', 0)
        if reviews >= 100:
            score += 30
        elif reviews >= 50:
            score += 20
        elif reviews >= 20:
            score += 15
        elif reviews >= 5:
            score += 10

        # Instagram вовлечённость (30 баллов)
        avg_likes = row.get('instagram_avg_likes', 0)
        if avg_likes >= 500:
            score += 30
        elif avg_likes >= 200:
            score += 20
        elif avg_likes >= 100:
            score += 15
        elif avg_likes >= 50:
            score += 10

        return min(score, 100)

    def score_occupancy_potential(self, row):
        """
        Оценка потенциала заполняемости (0-100)

        Косвенные индикаторы:
        - Популярность (высокий спрос)
        - Активность в соцсетях (маркетинг)
        - Наличие информации о ценах/услугах
        """
        score = 50  # Базовый балл

        # Высокий рейтинг = высокий спрос (+30)
        rating = row.get('rating_2gis', 0)
        if pd.notna(rating) and rating >= 4.0:
            score += 30

        # Активный Instagram = активный маркетинг (+20)
        posts = row.get('instagram_posts_last_month', 0)
        if posts >= 10:
            score += 20
        elif posts >= 5:
            score += 10

        return min(score, 100)

    def score_target_audience(self, row):
        """
        Оценка соответствия целевой аудитории (0-100)

        Анализирует по категориям и описанию:
        - Премиум сегмент
        - Эко-туризм
        - Семейный отдых
        - Молодёжный
        """
        score = 50  # Базовый балл (нейтральная оценка)

        # Анализируем категории
        categories = str(row.get('categories', '')).lower()
        description = str(row.get('ai_description', '')).lower()
        combined_text = categories + " " + description

        # Премиум индикаторы (+30)
        premium_keywords = ['премиум', 'люкс', 'vip', 'элитный', 'spa', 'wellness']
        if any(word in combined_text for word in premium_keywords):
            score += 30

        # Семейный отдых (+20)
        family_keywords = ['семейный', 'детский', 'family', 'kids', 'playground']
        if any(word in combined_text for word in family_keywords):
            score += 20

        return min(score, 100)

    def score_commercial_potential(self, row):
        """
        Оценка коммерческого потенциала (0-100)

        Индикаторы готовности к сотрудничеству:
        - Профессиональное присутствие в сети
        - Полнота контактной информации
        - Активное продвижение
        """
        score = 0

        # Профессиональный сайт (40 баллов)
        if pd.notna(row.get('website')) and row['website']:
            score += 40

        # Полные контакты (30 баллов)
        if pd.notna(row.get('email')) and row['email']:
            score += 20
        if pd.notna(row.get('phone')) and row['phone']:
            score += 10

        # Активное продвижение в Instagram (30 баллов)
        followers = row.get('instagram_followers', 0)
        posts = row.get('instagram_posts_last_month', 0)

        if followers >= 1000 and posts >= 5:
            score += 30
        elif followers >= 500 or posts >= 3:
            score += 15

        return min(score, 100)

    def calculate_total_score(self, row):
        """
        Рассчитывает итоговый score (0-100)
        """
        scores = {
            'online_activity': self.score_online_activity(row),
            'data_completeness': self.score_data_completeness(row),
            'popularity': self.score_popularity(row),
            'occupancy_potential': self.score_occupancy_potential(row),
            'target_audience': self.score_target_audience(row),
            'commercial_potential': self.score_commercial_potential(row)
        }

        # Взвешенная сумма
        total = sum(scores[key] * self.weights[key] for key in scores)

        return round(total, 2), scores

    def classify_organization(self, score):
        """
        Классифицирует организацию: Hot / Warm / Cold
        """
        if score >= 70:
            return "Hot"
        elif score >= 50:
            return "Warm"
        else:
            return "Cold"

    def process_dataframe(self, input_csv, output_csv):
        """
        Обрабатывает CSV и добавляет оценки
        """
        print("📊 Запуск системы оценки организаций...")

        if not os.path.exists(input_csv):
            print(f"❌ Файл {input_csv} не найден!")
            return None

        df = pd.read_csv(input_csv)
        print(f"✅ Загружено {len(df)} организаций\n")

        # Создаём новые колонки
        df['score_online_activity'] = 0.0
        df['score_data_completeness'] = 0.0
        df['score_popularity'] = 0.0
        df['score_occupancy_potential'] = 0.0
        df['score_target_audience'] = 0.0
        df['score_commercial_potential'] = 0.0
        df['total_score'] = 0.0
        df['category'] = ''

        print("🔍 Оценка организаций...\n")

        # Оцениваем каждую организацию
        for idx, row in df.iterrows():
            total_score, component_scores = self.calculate_total_score(row)
            category = self.classify_organization(total_score)

            # Записываем результаты
            df.at[idx, 'score_online_activity'] = component_scores['online_activity']
            df.at[idx, 'score_data_completeness'] = component_scores['data_completeness']
            df.at[idx, 'score_popularity'] = component_scores['popularity']
            df.at[idx, 'score_occupancy_potential'] = component_scores['occupancy_potential']
            df.at[idx, 'score_target_audience'] = component_scores['target_audience']
            df.at[idx, 'score_commercial_potential'] = component_scores['commercial_potential']
            df.at[idx, 'total_score'] = total_score
            df.at[idx, 'category'] = category

            print(f"  {'🔥' if category=='Hot' else '🌡️' if category=='Warm' else '❄️'} "
                  f"[{idx+1}/{len(df)}] {row.get('name', 'N/A')[:40]:40s} | "
                  f"Score: {total_score:5.1f} | {category}")

        # Сортируем по score
        df = df.sort_values('total_score', ascending=False)

        # Сохраняем
        df.to_csv(output_csv, index=False, encoding='utf-8')

        # Статистика
        print(f"\n{'='*60}")
        print(f"✅ Оценка завершена!")
        print(f"{'='*60}")
        print(f"\n📊 Распределение по категориям:")
        print(f"   🔥 Hot:  {len(df[df['category']=='Hot'])} организаций (score ≥ 70)")
        print(f"   🌡️ Warm: {len(df[df['category']=='Warm'])} организаций (score 50-69)")
        print(f"   ❄️ Cold: {len(df[df['category']=='Cold'])} организаций (score < 50)")

        print(f"\n📈 Средние баллы по критериям:")
        print(f"   • Online Activity:        {df['score_online_activity'].mean():.1f}/100")
        print(f"   • Data Completeness:      {df['score_data_completeness'].mean():.1f}/100")
        print(f"   • Popularity:             {df['score_popularity'].mean():.1f}/100")
        print(f"   • Occupancy Potential:    {df['score_occupancy_potential'].mean():.1f}/100")
        print(f"   • Target Audience:        {df['score_target_audience'].mean():.1f}/100")
        print(f"   • Commercial Potential:   {df['score_commercial_potential'].mean():.1f}/100")

        print(f"\n🏆 ТОП-5 организаций:")
        for i, row in df.head(5).iterrows():
            print(f"   {row['total_score']:5.1f} | {row['name'][:50]}")

        print(f"\n💾 Результат сохранён в: {output_csv}")

        return df


# 🎯 ЗАПУСК ОЦЕНКИ
scorer = OrganizationScorer()

graded_data = scorer.process_dataframe(
    input_csv="/content/final_instagram_enriched.csv",  # После Description Merger
    output_csv="/content/graded_organizations.csv"     # Финальный файл с оценками
)

print("\n✨ Готово! Переходим к последнему этапу: Outreach Email Sender")

📊 Запуск системы оценки организаций...
✅ Загружено 20 организаций

🔍 Оценка организаций...

  ❄️ [1/20] улица Мустафина, 1а/1                    | Score:  15.5 | Cold
  ❄️ [2/20] Tau-Asu                                  | Score:  22.8 | Cold
  ❄️ [3/20] База отдыха Tau Asu                      | Score:  15.5 | Cold
  ❄️ [4/20] Eco-hotel AQBULAQ                        | Score:  22.8 | Cold
  ❄️ [5/20] База отдыха Eco-Hotel AQBULAQ            | Score:  15.5 | Cold
  ❄️ [6/20] Гора Глэмпинг                            | Score:  25.2 | Cold
  ❄️ [7/20] База отдыха Гора Глэмпинг                | Score:  15.5 | Cold
  ❄️ [8/20] E`den                                    | Score:  27.2 | Cold
  ❄️ [9/20] База отдыха E`den                        | Score:  15.5 | Cold
  ❄️ [10/20] Roza eco village                         | Score:  34.2 | Cold
  ❄️ [11/20] улица Алма-Арасан Ущелье, 1/3            | Score:  15.5 | Cold
  ❄️ [12/20] Tau-Asu                                  | Score:  22.8 | Cold
  ❄️ 

### Email bot

In [ ]:
# @title Outreach Response & Email Sender (Unified)
import pandas as pd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import time
import os
from datetime import datetime

class OutreachEmailBot:
    """
    Автоматизированная система для генерации и отправки персонализированных писем
    """

    def __init__(self, sender_email, app_password, model_name="qwen3:14b"):
        """
        Инициализация бота

        Args:
            sender_email: Email отправителя (например, Gmail)
            app_password: App Password для Gmail (НЕ обычный пароль!)
            model_name: Модель Ollama для генерации писем
        """
        self.sender_email = sender_email
        self.app_password = app_password
        self.model_name = model_name

        # Шаблоны писем по категориям
        self.templates = {
            'Hot': """Здравствуйте, команда {name}!

Меня зовут [Ваше имя], я представляю mytravel.kz — ведущую платформу для поиска мест отдыха в Казахстане.

Мы заметили ваш {type} с превосходным рейтингом {rating} и активным присутствием в социальных сетях ({followers} подписчиков в Instagram). Ваша вовлечённость аудитории ({engagement}% engagement rate) впечатляет!

🎯 Почему mytravel.kz?
• 15,000+ активных туристов ежемесячно
• Премиум размещение для TOP объектов
• Система управления бронированиями
• Маркетинговая поддержка и продвижение

Мы готовы предложить вам ПРИОРИТЕТНОЕ размещение на главной странице и персональную поддержку менеджера.

Можем обсудить детали на этой неделе?

С уважением,
Команда mytravel.kz
📧 {sender_email}
🌐 https://mytravel.kz""",

            'Warm': """Добрый день, {name}!

Я представляю mytravel.kz — крупнейшую платформу бронирования отдыха в Казахстане.

Ваш {type} попал в наше поле зрения благодаря хорошим отзывам ({rating} на 2GIS) и активности в соцсетях. Мы помогаем таким объектам, как ваш, увеличивать загруженность на 30-50%.

💡 Что мы предлагаем:
• Доступ к 15,000+ туристов ежемесячно
• Удобная система бронирований
• Повышение видимости в поиске
• Бесплатное размещение на старте

Интересно узнать подробнее? Давайте созвонимся!

С уважением,
Команда mytravel.kz
📧 {sender_email}""",

            'Cold': """Здравствуйте!

Меня зовут [Ваше имя] из mytravel.kz — платформы для туристических объектов в Казахстане.

Мы заметили ваш объект {name} и хотели бы предложить сотрудничество. Наша платформа помогает местам отдыха увеличивать поток гостей и упрощать процесс бронирования.

🌟 Преимущества:
• Тысячи активных туристов
• Простая регистрация
• Первый месяц БЕСПЛАТНО

Если интересно — напишите, расскажу подробнее!

Команда mytravel.kz
📧 {sender_email}"""
        }

    def generate_personalized_message(self, row):
        """
        Генерирует персонализированное письмо на основе данных организации

        Args:
            row: Строка DataFrame с данными

        Returns:
            str: Готовое письмо
        """
        category = row.get('category', 'Cold')
        template = self.templates.get(category, self.templates['Cold'])

        # Подготавливаем данные для шаблона
        name = row.get('name', 'там')

        # Определяем тип места
        categories_str = str(row.get('categories', '')).lower()
        if 'глэмпинг' in categories_str or 'glamping' in categories_str:
            place_type = 'глэмпинг'
        elif 'база отдыха' in categories_str:
            place_type = 'база отдыха'
        elif 'отель' in categories_str or 'hotel' in categories_str:
            place_type = 'отель'
        elif 'курорт' in categories_str or 'resort' in categories_str:
            place_type = 'курорт'
        else:
            place_type = 'объект'

        # Данные для персонализации
        rating = row.get('rating_2gis', 'высокий рейтинг')
        if pd.notna(rating) and isinstance(rating, (int, float)):
            rating = f"{rating}/5.0"

        followers = row.get('instagram_followers', 0)
        if followers > 1000:
            followers_str = f"{followers:,}"
        else:
            followers_str = "активная аудитория"

        engagement = row.get('instagram_engagement_rate', 0)
        if pd.notna(engagement) and engagement > 0:
            engagement_str = f"{engagement}"
        else:
            engagement_str = "хорошая"

        # Заполняем шаблон
        message = template.format(
            name=name,
            type=place_type,
            rating=rating,
            followers=followers_str,
            engagement=engagement_str,
            sender_email=self.sender_email
        )

        return message

    def send_email(self, recipient_email, subject, body):
        """
        Отправляет email через SMTP Gmail

        Args:
            recipient_email: Email получателя
            subject: Тема письма
            body: Текст письма

        Returns:
            bool: True если успешно, False при ошибке
        """
        try:
            # Создаём сообщение
            msg = MIMEMultipart()
            msg['From'] = self.sender_email
            msg['To'] = recipient_email
            msg['Subject'] = subject
            msg.attach(MIMEText(body, 'plain', 'utf-8'))

            # Подключаемся к SMTP
            server = smtplib.SMTP('smtp.gmail.com', 587)
            server.starttls()
            server.login(self.sender_email, self.app_password)

            # Отправляем
            server.send_message(msg)
            server.quit()

            return True

        except Exception as e:
            print(f"      ❌ Ошибка отправки: {e}")
            return False

    def process_and_send(self, input_csv, output_csv,
                        send_to_categories=['Hot', 'Warm'],
                        max_emails=None,
                        delay_between_emails=5,
                        test_mode=False):
        """
        Обрабатывает CSV, генерирует и отправляет письма

        Args:
            input_csv: Входной CSV с оценками
            output_csv: Выходной CSV с результатами
            send_to_categories: Каким категориям отправлять ['Hot', 'Warm', 'Cold']
            max_emails: Максимум писем (None = без ограничений)
            delay_between_emails: Задержка между отправками (сек)
            test_mode: Если True, только генерирует письма без отправки
        """
        print("📧 Запуск Outreach Email Bot...")
        print(f"{'='*60}\n")

        if not os.path.exists(input_csv):
            print(f"❌ Файл {input_csv} не найден!")
            return None

        df = pd.read_csv(input_csv)
        print(f"✅ Загружено {len(df)} организаций")

        # Фильтруем по категориям
        df_filtered = df[df['category'].isin(send_to_categories)]
        print(f"🎯 Отправка письмам категорий: {', '.join(send_to_categories)}")
        print(f"📊 Организаций для обработки: {len(df_filtered)}\n")

        if max_emails:
            df_filtered = df_filtered.head(max_emails)
            print(f"⚠️ Ограничение: максимум {max_emails} писем\n")

        # Создаём колонки для результатов
        df['email_generated'] = ''
        df['email_sent'] = False
        df['email_sent_date'] = ''
        df['email_status'] = ''

        # Статистика
        generated = 0
        sent_success = 0
        sent_failed = 0
        skipped_no_email = 0

        subject = "Сотрудничество с mytravel.kz — топовая платформа путешествий"

        print(f"{'='*60}")
        print(f"{'🧪 ТЕСТОВЫЙ РЕЖИМ' if test_mode else '📤 ОТПРАВКА ПИСЕМ'}")
        print(f"{'='*60}\n")

        for idx, row in df_filtered.iterrows():
            org_name = row.get('name', 'N/A')
            org_category = row.get('category', 'N/A')
            org_score = row.get('total_score', 0)
            recipient_email = row.get('email')

            print(f"📨 [{generated+1}/{len(df_filtered)}] {org_name[:40]}")
            print(f"    Category: {org_category} | Score: {org_score:.1f}")

            # Проверяем наличие email
            if pd.isna(recipient_email) or not recipient_email or recipient_email == '':
                print(f"    ⚠️ Email не найден - пропуск\n")
                df.at[idx, 'email_status'] = 'No email'
                skipped_no_email += 1
                continue

            # Генерируем письмо
            try:
                email_body = self.generate_personalized_message(row)
                df.at[idx, 'email_generated'] = email_body
                generated += 1

                print(f"    ✅ Письмо сгенерировано")
                print(f"    📧 Получатель: {recipient_email}")

                # Отправляем (если не тестовый режим)
                if not test_mode:
                    success = self.send_email(recipient_email, subject, email_body)

                    if success:
                        df.at[idx, 'email_sent'] = True
                        df.at[idx, 'email_sent_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        df.at[idx, 'email_status'] = 'Sent successfully'
                        sent_success += 1
                        print(f"    📤 Отправлено успешно!")
                    else:
                        df.at[idx, 'email_sent'] = False
                        df.at[idx, 'email_status'] = 'Failed to send'
                        sent_failed += 1
                        print(f"    ❌ Ошибка отправки")

                    # Задержка между отправками
                    if generated < len(df_filtered):
                        print(f"    ⏱️ Задержка {delay_between_emails} сек...")
                        time.sleep(delay_between_emails)
                else:
                    df.at[idx, 'email_status'] = 'Test mode - not sent'
                    print(f"    🧪 Тестовый режим - не отправлено")

                print()

            except Exception as e:
                print(f"    ❌ Ошибка генерации: {e}\n")
                df.at[idx, 'email_status'] = f'Error: {str(e)}'
                sent_failed += 1

        # Сохраняем результат
        df.to_csv(output_csv, index=False, encoding='utf-8')

        # Итоговая статистика
        print(f"\n{'='*60}")
        print(f"✅ ЗАВЕРШЕНО!")
        print(f"{'='*60}")
        print(f"\n📊 Статистика:")
        print(f"   • Всего организаций: {len(df)}")
        print(f"   • Обработано: {len(df_filtered)}")
        print(f"   • Писем сгенерировано: {generated}")
        if not test_mode:
            print(f"   • Отправлено успешно: {sent_success}")
            print(f"   • Ошибок отправки: {sent_failed}")
        print(f"   • Пропущено (нет email): {skipped_no_email}")

        print(f"\n📈 По категориям:")
        for cat in ['Hot', 'Warm', 'Cold']:
            count = len(df[df['category'] == cat])
            sent = len(df[(df['category'] == cat) & (df['email_sent'] == True)])
            if count > 0:
                emoji = '🔥' if cat == 'Hot' else '🌡️' if cat == 'Warm' else '❄️'
                print(f"   {emoji} {cat}: {count} организаций | Отправлено: {sent}")

        print(f"\n💾 Результат сохранён в: {output_csv}")

        # Показываем примеры писем
        if generated > 0:
            print(f"\n📋 Пример сгенерированного письма:\n")
            print(f"{'='*60}")
            sample = df[df['email_generated'] != ''].iloc[0]
            print(f"Получатель: {sample.get('name')}")
            print(f"Email: {sample.get('email')}")
            print(f"Категория: {sample.get('category')}")
            print(f"\n{sample['email_generated']}")
            print(f"{'='*60}")

        return df


# 🎯 НАСТРОЙКА И ЗАПУСК

# ✅ Ваши данные для отправки
SENDER_EMAIL = "amina.baimakhanova@nu.edu.kz"
APP_PASSWORD = "vfrzotxhdnqlniki"  # App Password от Gmail

# Инициализация бота
bot = OutreachEmailBot(
    sender_email=SENDER_EMAIL,
    app_password=APP_PASSWORD,
    model_name="qwen3:14b"
)

# ВАРИАНТ 1: ТЕСТОВЫЙ РЕЖИМ (только генерация, без отправки)
print("🧪 Запуск в ТЕСТОВОМ РЕЖИМЕ (письма НЕ отправляются)\n")
result = bot.process_and_send(
    input_csv="/content/graded_organizations.csv",
    output_csv="/content/final_with_emails.csv",
    send_to_categories=['Hot', 'Warm'],  # Каким категориям отправлять
    max_emails=5,                         # Ограничение для теста
    delay_between_emails=5,               # Задержка между отправками
    test_mode=True                        # TRUE = только генерация
)

# ВАРИАНТ 2: РЕАЛЬНАЯ ОТПРАВКА (раскомментируйте после проверки)
# print("\n📤 Запуск РЕАЛЬНОЙ ОТПРАВКИ писем!\n")
# result = bot.process_and_send(
#     input_csv="/content/graded_organizations.csv",
#     output_csv="/content/final_with_emails.csv",
#     send_to_categories=['Hot', 'Warm'],  # Только Hot и Warm
#     max_emails=None,                      # None = без ограничений
#     delay_between_emails=10,              # 10 сек между письмами
#     test_mode=False                       # FALSE = реальная отправка
# )

print("\n\n🎉 ВСЕ МОДУЛИ ЗАВЕРШЕНЫ!")
print("="*60)
print("✅ Финальный файл: /content/final_with_emails.csv")
print("\n📊 Колонки в финальном CSV:")
print("   • Все данные из 2GIS")
print("   • Instagram метрики")
print("   • AI описание (ai_description)")
print("   • Оценки по всем критериям")
print("   • total_score и category (Hot/Warm/Cold)")
print("   • email_generated (текст письма)")
print("   • email_sent (статус отправки)")
print("   • email_sent_date (дата/время)")
print("   • email_status (результат)")
print("="*60)

🧪 Запуск в ТЕСТОВОМ РЕЖИМЕ (письма НЕ отправляются)

📧 Запуск Outreach Email Bot...

✅ Загружено 20 организаций
🎯 Отправка письмам категорий: Hot, Warm
📊 Организаций для обработки: 0

⚠️ Ограничение: максимум 5 писем

🧪 ТЕСТОВЫЙ РЕЖИМ


✅ ЗАВЕРШЕНО!

📊 Статистика:
   • Всего организаций: 20
   • Обработано: 0
   • Писем сгенерировано: 0
   • Пропущено (нет email): 0

📈 По категориям:
   ❄️ Cold: 20 организаций | Отправлено: 0

💾 Результат сохранён в: /content/final_with_emails.csv


🎉 ВСЕ МОДУЛИ ЗАВЕРШЕНЫ!
✅ Финальный файл: /content/final_with_emails.csv

📊 Колонки в финальном CSV:
   • Все данные из 2GIS
   • Instagram метрики
   • AI описание (ai_description)
   • Оценки по всем критериям
   • total_score и category (Hot/Warm/Cold)
   • email_generated (текст письма)
   • email_sent (статус отправки)
   • email_sent_date (дата/время)
   • email_status (результат)
